In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [3]:
pw = "Wtcantfw36c!"

In [4]:


DATABASE_CONFIG = {
    'host': 'dandyweb01fl',
    'database': 'aedna_metadata_test',
    'port': '5432',
    'user': 'upload_user',
    'password': pw,
    'schema_name': 'test_1'
}

DATABASE_CONFIG = {
    'host': 'dandyweb01fl',
    'database': 'aedna_metadata_test',
    'port': '5432',
    'user': 'upload_user',
    'password': pw,
    'schema_name': 'test_1'
}

DATABASE_CONFIG_2 = {
    'host': DATABASE_CONFIG['host'],
    'dbname': DATABASE_CONFIG['database'],
    'port': DATABASE_CONFIG['port'],
    'user': DATABASE_CONFIG['user'],
    'password': DATABASE_CONFIG['password'],
}

DATABASE_CONFIG_READ_ONLY = {
    'host': DATABASE_CONFIG['host'],
    'dbname': DATABASE_CONFIG['database'],
    'port': DATABASE_CONFIG['port'],
    'user': 'read_user',
    'password': DATABASE_CONFIG['password'],
}

ENGINE = create_engine(
    f"postgresql://{DATABASE_CONFIG['user']}:{DATABASE_CONFIG['password']}@{DATABASE_CONFIG['host']}:{DATABASE_CONFIG['port']}/{DATABASE_CONFIG['database']}")

DATABASE_CONFIG_READ_ONLY = {
    'host': DATABASE_CONFIG['host'],
    'dbname': DATABASE_CONFIG['database'],
    'port': DATABASE_CONFIG['port'],
    'user': 'read_user',
    'password': DATABASE_CONFIG['password'],
}

ENGINE = create_engine(
    f"postgresql://{DATABASE_CONFIG_READ_ONLY['user']}:{DATABASE_CONFIG_READ_ONLY['password']}@{DATABASE_CONFIG_READ_ONLY['host']}:{DATABASE_CONFIG_READ_ONLY['port']}/{DATABASE_CONFIG_READ_ONLY['dbname']}")

In [5]:
pd.set_option('display.max_rows', 1000)  # Set maximum number of rows to display
pd.set_option('display.max_columns', 1000)

In [6]:
wldf_q = "select * from test_1.edna_wetlab_report;"
asdf_q = "select * from test_1.edna_archive_sample;"
cgg_q = "select * from super_simple_db.cgg_sediment_water;"

In [7]:
core_ids = ["ISL2020_05_25", "ISL2020_09_70", "ISL22_35.8", "ISL22_40.4", "ISL22_49.3b"]

In [8]:
wldf = pd.read_sql(wldf_q, dtype=str, con=ENGINE)
asdf = pd.read_sql(asdf_q, dtype=str, con=ENGINE)
cgg = pd.read_sql(cgg_q, dtype=str, con=ENGINE)

Cleaning the data so I can do proper filter and merges

In [9]:
# TODO: Make sure to check that replacemants doesnt result in duplicate data
core_ids = list(map(lambda x: x.upper(), core_ids))
core_ids = list(map(lambda x: x.replace(" ", ""), core_ids))
core_ids = list(map(lambda x: x.replace("," , "."), core_ids))
core_ids = list(map(lambda x: x.replace("-" , "_"), core_ids))
asdf["BulkSampleID"] = asdf["BulkSampleID"].str.strip()
asdf["BulkSampleID"] = asdf["BulkSampleID"].str.replace("-", "_")
asdf["BulkSampleID"] = asdf["BulkSampleID"].str.replace(",", ".")
asdf["BulkSampleID"] = asdf["BulkSampleID"].str.upper()
asdf["BulkSampleID"] = asdf["BulkSampleID"].str.replace(" ", "")
cgg["Museum ID/sample ID"] = cgg["Museum ID/sample ID"].str.strip()
cgg["Museum ID/sample ID"] = cgg["Museum ID/sample ID"].str.replace("," , ".")
cgg["Museum ID/sample ID"] = cgg["Museum ID/sample ID"].str.replace("-" , "_")
cgg["Museum ID/sample ID"] = cgg["Museum ID/sample ID"].str.upper()
cgg["Museum ID/sample ID"] = cgg["Museum ID/sample ID"].str.replace(" ", "")
cgg["CGG ID"] = cgg["CGG ID"].str.strip()
cgg["CGG ID"] = cgg["CGG ID"].str.replace("-" , "_")
cgg["CGG ID"] = cgg["CGG ID"].str.upper()
cgg["CGG ID"] = cgg["CGG ID"].str.replace(" ", "")
wldf["Archive Sample ID"] = wldf["Archive Sample ID"].str.strip()
wldf["Archive Sample ID"] = wldf["Archive Sample ID"].str.replace("-", "_")
wldf["Archive Sample ID"] = wldf["Archive Sample ID"].str.replace("," , ".")
wldf["Archive Sample ID"] = wldf["Archive Sample ID"].str.replace(" ", "")
wldf["Archive Sample ID"] = wldf["Archive Sample ID"].str.upper()

Getting all rows from cgg and jesper data where core id starts with the master core ids from Kurt

In [10]:
cores_kurt_cgg = cgg[cgg["Museum ID/sample ID"].isin(core_ids)]
cores_kurt_asdf = asdf[asdf["BulkSampleID"].isin(core_ids)]

Found 143 archive samples 

In [11]:
len(cores_kurt_asdf)

143

In [12]:
cores_kurt_asdf.head()

,ArchiveSampleID,PositionInRack,RackName,RackID,BulkSampleID,DepthSampledCalTape,DepthOrderedCalTape,OrganicContent,SurfaceExposed,RemarksArchiveSampling,SampledBy1,SampledBy2,SamplingDate,Submitter,SubmissionDate,RemarksSubmission,database_insert_by,from_spreadsheet,database_insert_datetime_utc,upload_uuid
2915,LV3005273838,C03,ARack0127,LVL500291932,ISL22_40.4,0.5,"0,5",None,None,None,FM,VSS,2022-10-07,WRF,2022-10-07,None,upload_user,SedimentArchive&SubSamplingALL_240417.txt,2024-04-30 10:37:27.491009+02:00,ef07c7b5-93a0-4223-ad48-6673ea022ccc
2916,LV3005272806,D03,ARack0127,LVL500291932,ISL22_40.4,1.5,"1,5",None,None,None,FM,VSS,2022-10-07,WRF,2022-10-07,None,upload_user,SedimentArchive&SubSamplingALL_240417.txt,2024-04-30 10:37:27.491009+02:00,ef07c7b5-93a0-4223-ad48-6673ea022ccc
2917,LV3005273689,A04,ARack0127,LVL500291932,ISL22_40.4,3.0,3,None,None,None,FM,VSS,2022-10-07,WRF,2022-10-07,None,upload_user,SedimentArchive&SubSamplingALL_240417.txt,2024-04-30 10:37:27.491009+02:00,ef07c7b5-93a0-4223-ad48-6673ea022ccc
2918,LV3005272805,B04,ARack0127,LVL500291932,ISL22_40.4,4.0,4,None,None,None,FM,VSS,2022-10-07,WRF,2022-10-07,None,upload_user,SedimentArchive&SubSamplingALL_240417.txt,2024-04-30 10:37:27.491009+02:00,ef07c7b5-93a0-4223-ad48-6673ea022ccc
2919,LV3005272792,C04,ARack0127,LVL500291932,ISL22_40.4,5.5,"5,5",None,None,None,FM,VSS,2022-10-07,WRF,2022-10-07,None,upload_user,SedimentArchive&SubSamplingALL_240417.txt,2024-04-30 10:37:27.491009+02:00,ef07c7b5-93a0-4223-ad48-6673ea022ccc


In [13]:
cores_kurt_cgg.head()

,CGG ID,Museum ID/sample ID,Lab no.,Stock sample left,Extraction#,Extraction date,Kind of Library,Library date,Intern Provider,Sample type,Depth,height (m) asl.,Material type,Sampled as,Sample size,Depositional environment,Age,Geological age,Site,Site info,City/town/location,State/province/region,Country,Continent,Lat,Lon,GPS,Sample Provider,Museum/Institution,Contact info,Date collected,Date recieved,Date returned,Sample location,Extraction location,In care of,Date out,Project name,Supervisor,Analyses history,GenBank ref#,Publications,References,Notes,Thawed up,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52,Unnamed: 53,Unnamed: 54,Unnamed: 57,Unnamed: 62,Unnamed: 74,database_insert_by,from_spreadsheet,database_insert_date_utc,uid
12660,CGG_3_012673,ISL2020_09_70,09 - 70,None,None,cm,70 cm,None,Kurt Kjær/Nicolaj Krog Larsen,sediment core,70,32.86,sediment,sub-sampled from cores,2 cm2,Lake,Viking Age?,Late Holocene,Hvaleyravanet,Lake depth 0.9 m,Hafnarförður,Höfuðþorgarsvæðið,Iceland,Island of Iceland,"64°02'210""N","21°55'464""W",33.56 m,Kurt Kjær/Nicolaj Krog Larsen,Globe:GeoGen,None,2020-08-15 00:00:00,2020-09-10 00:00:00,None,Walk-in freezer,None,None,Icelandic lakes,Kurt Kjær,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,glj523,CGG_Database_Sediment_Water_231115_PSO.xlsx,2024-01-08 15:41:51.672002+01:00,dc934619-5325-46a5-a010-cb1ce9334a88
12691,CGG_3_012703,ISL2020_05_25,05 - 25,None,None,cm,25 cm,None,Kurt Kjær/Nicolaj Krog Larsen,sediment core,25,91.75,sediment,sub-sampled from cores,2 cm2,Lake,Viking Age?,Late Holocene,Oddnyjartjörn,Lake depth 1.5 m,Skeiðflötur,Suðurland,Iceland,Island of Iceland,"63°26'50.2""N","19°10'13.0""W",92 m,Kurt Kjær/Nicolaj Krog Larsen,Globe:GeoGen,None,2020-08-15 00:00:00,2020-09-10 00:00:00,None,Walk-in freezer,None,None,Icelandic lakes,Kurt Kjær,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,glj523,CGG_Database_Sediment_Water_231115_PSO.xlsx,2024-01-08 15:41:51.672002+01:00,5b8f0e85-4b3e-4227-93f2-999a4a209d14


In [14]:
cores_kurt_cgg["Museum ID/sample ID"].unique()

array(['ISL2020_09_70', 'ISL2020_05_25'], dtype=object)

In [15]:
cores_kurt_asdf["BulkSampleID"].unique()

array(['ISL22_40.4', 'ISL22_35.8'], dtype=object)

In [16]:
len(cores_kurt_asdf["BulkSampleID"].unique())

2

In [17]:
def find_duplicates(lst):
    seen = {}
    duplicates = []

    for item in lst:
        if item in seen:
            duplicates.append(item)
        else:
            seen[item] = 1

    return duplicates

Finding 1 duplicate core ID that is present in both Jespers and CGG:

In [18]:
my_list = list(cores_kurt_asdf["BulkSampleID"].unique()) + list(cores_kurt_cgg["Museum ID/sample ID"].unique())
find_duplicates(my_list)

[]

##### Merging
Merging on Archive Sample ID. Caution: Xihan says this ID is not robust in her data, so maybe this is error prone? Think it should be fine as long as its LV codes however. Best would be to merge with Robot Sample ID, will do so when I get the data from Jesper.

In [35]:
merged_on_aID = pd.merge(cores_kurt_asdf, wldf, left_on='ArchiveSampleID', right_on='Archive Sample ID', how='inner')

In [36]:
merged_on_aID_essentials = merged_on_aID[['BulkSampleID', "ArchiveSampleID", "Robot Sample ID", "Library ID", 'FastQ File ID', "DepthSampledCalTape"]]

In [37]:
len(merged_on_aID)

37

In [38]:
merged_on_aID_essentials.head()

,BulkSampleID,ArchiveSampleID,Robot Sample ID,Library ID,FastQ File ID,DepthSampledCalTape
0,ISL22_40.4,LV3005350770,LV6000200096,LV7008960500,LV7008960500-LV6000200096-LV3005350770,35.0
1,ISL22_40.4,LV3005350784,LV6000200088,LV7008960512,LV7008960512-LV6000200088-LV3005350784,38.0
2,ISL22_40.4,LV3005350792,LV6000200127,LV7008960524,LV7008960524-LV6000200127-LV3005350792,42.5
3,ISL22_40.4,LV3005350786,LV6000200119,LV7008960536,LV7008960536-LV6000200119-LV3005350786,45.5
4,ISL22_40.4,LV3005350771,LV6000200111,LV7008960451,LV7008960451-LV6000200111-LV3005350771,48.5


Merging CGG number with Wetlab report

In [39]:
merged_on_CGG_ID = pd.merge(cores_kurt_cgg, wldf, left_on='CGG ID', right_on='Archive Sample ID', how='inner')

In [40]:
merged_on_CGG_ID_essentials = merged_on_CGG_ID[["Museum ID/sample ID", 'CGG ID', "Library ID", 'FastQ File ID', "Depth", "height (m) asl.", "Age", "Geological age", "Lat", "Lon", "GPS"]]

In [41]:
len(merged_on_CGG_ID)

0

Merging Core segment id (field sample ID) with wet lab report (shouldnt work)

In [42]:
merged_on_museum_id = pd.merge(cores_kurt_cgg, wldf, left_on='Museum ID/sample ID', right_on='Archive Sample ID', how='inner')

In [43]:
merged_on_museum_id_essentials = merged_on_museum_id[["Museum ID/sample ID", 'CGG ID', "Library ID", 'FastQ File ID', "Depth", "height (m) asl.", "Age", "Geological age", "Lat", "Lon", "GPS"]]

In [44]:
len(merged_on_museum_id)

0

In [45]:
merged_on_bulksampleid = pd.merge(cores_kurt_asdf, wldf, left_on='BulkSampleID', right_on='Archive Sample ID', how='inner')

In [46]:
merged_on_bulksampleid_essentials = merged_on_bulksampleid[['BulkSampleID', "ArchiveSampleID", "Robot Sample ID", "Library ID", 'FastQ File ID', "DepthSampledCalTape"]]

In [47]:
len(merged_on_bulksampleid_essentials)

0

In [48]:
merged_on_bulksampleid_essentials

,BulkSampleID,ArchiveSampleID,Robot Sample ID,Library ID,FastQ File ID,DepthSampledCalTape


In [49]:
cgg_essential = cgg[["Museum ID/sample ID", 'CGG ID', "Depth", "height (m) asl.", "Age", "Geological age", "Lat", "Lon", "GPS"]]
pd.set_option('future.no_silent_downcasting', True)
# Full 
merged_on_aID = merged_on_aID.rename(columns={"BulkSampleID": "FieldSampleID"})
merged_on_bulksampleid = merged_on_bulksampleid.rename(columns={"BulkSampleID": "FieldSampleID"})
merged_on_CGG_ID.loc[:, "FieldSampleID"] = merged_on_CGG_ID.loc[:, "Museum ID/sample ID"]
merged_on_museum_id.loc[:, "FieldSampleID"] = merged_on_museum_id.loc[:, "Museum ID/sample ID"]

conc = pd.concat([merged_on_aID, merged_on_CGG_ID])
conc = conc.reset_index()
merged_df = pd.merge(conc, cgg, left_on="FieldSampleID", right_on="CGG ID", how='left')
# Now replace NaN values in common columns of df1 with values from df2
for column in conc.columns:
    if column in cgg.columns:
        merged_df[column+'_x'] = merged_df[column+'_x'].fillna(merged_df[column+'_y']).infer_objects(copy=False)
        merged_df = merged_df.drop(columns=[column+'_y'])

# Rename the columns back to their original names
full_merged_df = merged_df.rename(columns={col: col[:-2] for col in merged_df.columns if col.endswith('_x')})

# Essential

# CLeaning
merged_on_aID_essentials = merged_on_aID_essentials.rename(columns={"BulkSampleID": "FieldSampleID"})
merged_on_bulksampleid_essentials = merged_on_bulksampleid_essentials.rename(columns={"BulkSampleID": "FieldSampleID"})
merged_on_CGG_ID_essentials.loc[:, "FieldSampleID"] = merged_on_CGG_ID_essentials.loc[:, "Museum ID/sample ID"]
merged_on_museum_id_essentials.loc[:, "FieldSampleID"] = merged_on_museum_id_essentials.loc[:, "Museum ID/sample ID"]

conc = pd.concat([merged_on_aID_essentials, merged_on_bulksampleid_essentials, merged_on_CGG_ID_essentials, merged_on_museum_id_essentials])
conc = conc.reset_index()
merged_df = pd.merge(conc, cgg_essential, left_on="FieldSampleID", right_on="CGG ID", how='left')
for column in conc.columns:
    if column in cgg_essential.columns:
        merged_df[column+'_x'] = merged_df[column+'_x'].fillna(merged_df[column+'_y'])
        merged_df = merged_df.drop(columns=[column+'_y'])

# Rename the columns back to their original names
essential_merged_df = merged_df.rename(columns={col: col[:-2] for col in merged_df.columns if col.endswith('_x')})